In [1]:
import pandas as pd 
df=pd.read_excel('./카테고리재분류3.xlsx')
df[['distance','walking_time']]=0
df=df.rename(columns={'category':'category_all'})
df.to_excel('카테고리재분류3.xlsx',index=False)

In [2]:
pd.set_option('display.max_rows',None)
df['category_2'].value_counts()

category_2
육류,고기       262
치킨          206
해물,생선       167
고기구이        166
백반/가정식      157
분식          131
중국요리         99
한식면요리        78
국밥           76
파스타/스테이크     74
초밥/회         61
탕/찌개         58
족발/보쌈        48
순대           47
일식일반         46
돈까스          43
마라/양꼬치       42
떡볶이          40
햄버거          39
양식일반         30
고기요리         25
전/민속주점       20
간편식/덮밥       18
이탈리안         18
중식           18
죽            18
제과,베이커리      18
샤브샤브         17
찌개/탕         16
돈까스,우동       15
면요리(일식)      15
국수           14
샐러드          14
닭강정          13
베트남음식        11
양꼬치          10
일본식라면         9
멕시칸,브라질       9
해물요리/찜        9
컵밥            9
사철탕,영양탕       9
간식            8
냉면            8
초밥,롤          7
한식뷔페          7
태국음식          7
찌개/조림         7
찌개,전골         7
탕             6
동남아음식         6
도시락           6
두부전문점         6
건강식/정식        5
감자탕           5
한솥도시락         5
싸다김밥          4
토스트           4
쌈밥            4
피자            4
구내식당          4
식품            4
일식집          

In [3]:
df[df['category_2'] == '일식일반'] 

,name,address,category_all,latitude,longitude,category_1,category_2,distance,walking_time
161,고씨네 신림대학동점,서울 관악구 신림동 1515-4,음식점 > 일식,37.470458,126.936401,일식,일식일반,0,0
216,구루메키친,서울 관악구 봉천동 1620-30,음식점 > 일식,37.477953,126.958039,일식,일식일반,0,0
260,규이치,서울 관악구 봉천동 870-13,음식점 > 일식,37.481744,126.951451,일식,일식일반,0,0
426,당당,서울 관악구 남현동 1056-31,음식점 > 일식,37.476107,126.976133,일식,일식일반,0,0
466,덮스테이,서울 관악구 봉천동 1597-7,음식점 > 일식,37.478090,126.952740,일식,일식일반,0,0
480,도쿄시장 신림점,서울 관악구 신림동 1639-30,음식점 > 일식,37.483359,126.928412,일식,일식일반,0,0
505,동경산책,서울 관악구 봉천동 1604-8,음식점 > 일식,37.478870,126.954200,일식,일식일반,0,0
636,마츠,서울 관악구 신림동 1460-1,음식점 > 일식,37.486454,126.926089,일식,일식일반,0,0
696,멘부리 신림점,서울 관악구 봉천동 928-5,음식점 > 일식,37.481550,126.942508,일식,일식일반,0,0
934,봉천다찌,서울 관악구 봉천동 1612-2,음식점 > 일식,37.479396,126.955764,일식,일식일반,0,0


In [4]:
df.loc[df['name'].str.contains('돼지게티'), ['category_1', 'category_2']] = ['양식', '파스타']
print(df[df['name'].str.contains('룸타이')][['name', 'category_1', 'category_2']])

Empty DataFrame
Columns: [name, category_1, category_2]
Index: []


In [5]:
def refine_japanese_general(row):
    if row['category_2'] == '일식일반':
        name = str(row['name']).upper()
        
        # 1. 초밥/스시/회 전문
        if any(k in name for k in ['스시', 'SUSHI', '은행골', '사케앤살몬', '연어', '푸른연어', '혼맛']):
            return '초밥/스시'
            
        # 2. 덮밥/돈부리 전문
        if any(k in name for k in ['덮스테이', '멘부리', '스타동', '소미당', '앤미']):
            return '일식덮밥'
            
        # 3. 라멘/우동/소바 전문 (면요리)
        if any(k in name for k in ['호랑이면', '히토스지', '창운', '국수']):
            return '일식면요리'
            
        # 4. 돈까스/튀김 전문
        if any(k in name for k in ['텐동', '요츠야', '카츠', '이자와', '규카츠', '후추네']):
            return '돈까스/튀김'
            
        # 5. 야키토리/이자카야/술집 (저녁 위주)
        # 사장님, '술집'은 다 빼기로 했지만 '음식점'으로 분류된 이자카야는 일단 살려둡니다.
        if any(k in name for k in ['야키토리', '야나기', '야키니쿠', '마츠', '사케바', '이자카야', '봉천다찌', '요술사']):
            return '일식주점'
            
        # 6. 일본식 카레/스키야키 등 기타 일식
        if any(k in name for k in ['고씨네', '카레', '스키야키', '키요이', '스아게', '당당']):
            return '기타일식'

    return row['category_2']

# 적용!
df['category_2'] = df.apply(refine_japanese_general, axis=1)

In [ ]:
def refine_japanese_v2(row):
    if row['category_2'] == '일식일반':
        name = str(row['name'])
        
        # 1. 일식 덮밥/가정식 (샤로수길 핫플 위주)
        if any(k in name for k in ['킷사서울', '동경산책', '소쿠리', '테라다식당', '한술더']):
            return '일식덮밥/가정식'
            
        # 2. 이자카야/심야식당 (주점 성격이 강한 곳)
        if any(k in name for k in ['심야식당', '구루메키친', '이쿠', '주수', '아자스', '도쿄시장', '오지']):
            return '일식주점'
            
        # 3. 야키니쿠/야키토리/고기류
        if any(k in name for k in ['규이치', '혼야키', '타마']): # 타마는 오코노미야키/철판요리
            return '일식구이/철판'
            
        # 4. 장어/회 전문
        if any(k in name for k in ['우나기', '삿뽀로']):
            return '회/일식코스'

        # 5. 기타 특색 메뉴
        if '스윗보울' in name: return '샐러드/포케' # 일식 베이스 포케집인 경우
        if '카카라' in name: return '일식카레'

    return row['category_2']

# 적용
df['category_2'] = df.apply(refine_japanese_v2, axis=1)


,name,address,category_all,latitude,longitude,category_1,category_2,distance,walking_time
161,고씨네 신림대학동점,서울 관악구 신림동 1515-4,음식점 > 일식,37.470458,126.936401,일식,기타일식,0,0
426,당당,서울 관악구 남현동 1056-31,음식점 > 일식,37.476107,126.976133,일식,기타일식,0,0
1230,스아게 샤로수길,서울 관악구 봉천동 1599-4,음식점 > 일식,37.479204,126.953544,일식,기타일식,0,0
2041,키요이 스키야키,서울 관악구 봉천동 1620-5,음식점 > 일식,37.478457,126.957095,일식,기타일식,0,0


In [12]:
df[df['name'] == '아비꼬 서울대입구역점']

,name,address,category_all,latitude,longitude,category_1,category_2,distance,walking_time
1353,아비꼬 서울대입구역점,서울 관악구 봉천동 1601-1,음식점 > 퓨전요리 > 아비꼬,37.480393,126.954324,퓨전요리,아비꼬,0,0


In [13]:
df['category_2'].value_counts()

category_2
육류,고기       262
치킨          206
해물,생선       167
고기구이        166
백반/가정식      157
분식          131
중국요리         99
한식면요리        78
국밥           76
파스타/스테이크     74
초밥/회         61
탕/찌개         58
족발/보쌈        48
순대           47
돈까스          43
마라/양꼬치       42
떡볶이          40
햄버거          39
양식일반         30
고기요리         25
전/민속주점       20
간편식/덮밥       18
중식           18
죽            18
제과,베이커리      18
이탈리안         18
샤브샤브         17
찌개/탕         16
면요리(일식)      15
일식주점         15
돈까스,우동       15
샐러드          14
국수           14
닭강정          13
베트남음식        11
양꼬치          10
일본식라면         9
사철탕,영양탕       9
해물요리/찜        9
컵밥            9
멕시칸,브라질       9
간식            8
냉면            8
태국음식          7
찌개/조림         7
찌개,전골         7
한식뷔페          7
초밥,롤          7
동남아음식         6
도시락           6
두부전문점         6
탕             6
건강식/정식        5
일식덮밥/가정식      5
일식덮밥          5
한솥도시락         5
감자탕           5
식품            4
초밥/스시         4
쌈밥            4
싸다김밥          4
토스트          

In [ ]:
before_count = len(df)
df = df[df['category_2'] != '일식주점']
after_count = len(df)

print(f"일식주점 {before_count - after_count}개 삭제 완료!")

df.loc[df['name'].str.contains('고씨네', na=False), 'category_2'] = '일식카레'

print("\n변경 후 '고씨네' 상태:")
print(df[df['name'].str.contains('고씨네')][['name', 'category_2']])

일식주점 15개 삭제 완료!

변경 후 '고씨네' 상태:
           name category_2
161  고씨네 신림대학동점       일식카레


In [15]:
df.to_excel('카테고리_재분류4.xlsx')